
# Real-world data coding for neuroscientists (ReCoN)

### MSc in Translational Neuroscience,

### Department of Brain Sciences, Faculty of Medicine,

### Imperial College London

### Contributors: Anastasia Ilina, Cecilia Rodriguez, Marirena Bafaloukou, Katarzyna Marta Zoltowska, Rishideep Chatterjee, Sahar Rahbar, Cynthia Sandor

### Autumn 2025


# 🧠 Computing with Real‑World Medical Data in Python (MSc Translational Neuroscience)
## Tutorial 2 – Tabular data with pandas on Electronic Healthcare Records (EHR)

**Part 1: Data cleaning and preprocessing with pandas**






## Learning outcomes

**Overall aim**: develop skills for statistical analysis of longitudinal medical data.

**You will learn to:**
- Manipulate tabular data in `pandas` (indexing, filtering, grouping, reshaping).
- Pre‑clean longitudinal EHR data (duplicates, types, missingness, **data entry errors in medications**).
- Engineer features with functions and `.apply`.


## 📚 Supporting matetial - pandas cheat sheet
> This cheat sheet is a great reference for pandas methods:

> [Pandas Cheat Sheet](https://pandas.pydata.org/Pandas_Cheat_Sheet.pdf)


## 🎯 Foundations: pandas DataFrames (theory)


![pandas meme](https://i.programmerhumor.io/2025/03/bf92b0031ba27033fcf1534cd911e834c963afbc0c31b90fc9550eef0fafed92.png)

In [ ]:
import pandas as pd
import numpy as np

# We'll make a tiny mock DataFrame to play with
data = {
    "patient_id": ["P01", "P02", "P03", "P04", "P01", "P02", "P03", "P04"],
    "sex": ["Male", "Female", "Female", "Male", "Male", "Female", "Female", "Male"],
    "date_of_birth": ["14.05.1950", "22.08.1965", "30.11.1970", "15.01.1980",
                                  "14.05.1950", "22.08.1965", "30.11.1970", "15.01.1980"],
    "updrs_iii": [22, 30, 40, None, 25, 31, 40, None],     # missing value
    "ledd_mg": [300, 500, 800, 0, 500, 600, 850, 0],
    "visit_date": ["2023-01-01", "2023-01-02", "2023-01-03", "2023-01-04", "2024-02-08", "2025-02-23", "2023-10-06", "2024-06-04"],
    "medication": ["Co-Carbidopa 12.5/50", "Rotigotine 0.5", "   Co-BEneldopa;", "", "Co-carbidopa,,", "Rotigotine.  ", "LEVODOPA + ENTACAPONE", "N/A"],
    "other_medication": ['Yes', 'Yes', 'No', 'No', 'Yes', 'No', 'Yes', 'No'],
    'BMI': ['22.5', '25.0', '30.0', '28.0', '22.7', '24.0', '36.0', '27.6'],
    "id_last_update": ["09/2024", "01/2025", "12/2023", "09/2024", "09/2024", "01/2025", "12/2023", "09/2024"]
}
df = pd.DataFrame(data)
df




### 1. Selection and Filtering

**Concept:**
A **DataFrame** is a core `pandas` object — conceptually similar to an Excel sheet — but far more powerful for analysis.
It stores data in **rows** and **columns**, each with **labels** (names), and is ideal for working with structured data such as **Electronic Health Records (EHRs)**.

#### 🩺 **Why this matters in medical data**

EHRs are almost always stored in tabular form, making DataFrames a natural fit for biomedical and clinical research workflows.

* Each **row** typically represents a single **observation** or **encounter** — for example, a patient’s visit to a clinic.
* Each **column** represents a different **attribute** or **feature** of that record — such as patient ID, age, diagnosis, treatment, or visit date.

In our **synthetic Parkinson’s dataset**,
each row represents a **medication record** for a patient during one clinic visit.
Because each patient can have **multiple visits**, you will see several rows per patient, recorded at different time points.

#### 🧩 **What’s inside this dataset**

Alongside demographic information (e.g., **age at visit**, **sex**), the dataset includes several **clinical features**:

* **UPDRS III** — a clinical rating scale assessing the severity of motor symptoms in Parkinson’s disease.
* **LEDD (Levodopa Equivalent Daily Dose)** — a calculated metric expressing the total daily dose of dopaminergic medication, normalized across different drugs.

These variables will be used throughout the tutorial for cleaning, transformation, visualization, and statistical modeling.


#### 🔍 **Selecting and filtering in `pandas`**

You can access or subset DataFrames using **labels** (row/column names) or **logical conditions**:

* `df.loc[rows, columns]` → select by explicit labels
* `df.iloc[row_indices, col_indices]` → select by position (numeric index)
* `df.query("condition")` → filter rows using SQL-like syntax

This allows you to quickly focus on, for example:

* All visits for a particular patient
* Only those observations with a specific diagnosis or medication
* A subset of columns containing clinical or demographic features



In [ ]:
# Let's get a quick summary of the DataFrame first: its structure, data types, and non-null counts per column
df.info()

In [ ]:
# You can use .head() and .tail() to quickly inspect the first or last few rows of a DataFrame
df.head()  # shows first 5 rows by default

In [ ]:
df.tail(2)  # last 2 rows

In [ ]:
# Select columns
df[["patient_id", "visit_date", "updrs_iii"]]

In [ ]:
# you can overlay it with .head() or .tail()
df[["patient_id", "visit_date", "updrs_iii"]].head()

In [ ]:
# more ways to select columns
cols = ["patient_id", "visit_date", "updrs_iii"]
df[cols]


In [ ]:
# Row and column selection by label
df.loc[1, "sex"]                   # row with label 1 (might not be the first row) and the column named "sex"

In [ ]:
# Row and column selection by index position
df.iloc[0, 1]                       # first row (position 0) and the second column (position 1)

In [ ]:

# Select multiple rows/cols by label
df.loc[0:2, ["patient_id", "visit_date"]]

In [ ]:

# Conditional selection: only patients older than 60
df.loc[df["updrs_iii"] < 30, ["patient_id", "visit_date", "ledd_mg", "updrs_iii"]]

In [ ]:
# Conditional selection: only male patients
df.loc[df["sex"] == 'Male', ["patient_id",  "date_of_birth", "ledd_mg", "updrs_iii", "sex"]] 
# you can also use !=, >, <, >=, <=

In [ ]:
# Conditional selection: patients on medication
df.loc[df["medication"] != "None", ["patient_id", "visit_date", "medication"]]

In [ ]:
# you can also select columns using regex of the column names 

# let's select the columns that contain "date"
df.filter(regex="date")


In [ ]:

# not lets select the columns that end with "id"
df.filter(regex="id$")  # $ means "ends with"

In [ ]:
# now let's select the columns that start with "id"
df.filter(regex="^id")  # ^ means "starts with"



In [ ]:
# you can also combine multiple conditions using & (and), | (or), ~ (not)
# Note: use parentheses around each condition when combining


# let's select the column that starts with "id" and ends with "date"
df.filter(regex="^id.*date$")

In [ ]:
# you can overlay different methods as much as you want
# let's select the columns that contain "date" and only show the first 3 rows
df.loc[df["sex"] == 'Male', ["patient_id",  "date_of_birth", "ledd_mg", "updrs_iii", "sex"]].filter(regex = "date").head(1)

In [ ]:
# you can also call the columns using df.filter().columns
df.loc[df["ledd_mg"] < 300, df.filter(regex = "date").columns]

In [ ]:
# For those familiar with SQL, you can also use query() method for conditional selection
# Using query() (SQL-like)
df.query("updrs_iii < 30 and sex == 'Male'")


### 🍎 <span style="color:red; font-weight:bold">Student challenge 


 <span style="color:red; font-weight:bold">Now your turn!</span>

  <span style="color:red; font-weight:bold"> Use the dataset `df` to pratice filtering.</span>

**Tasks:**

1. Write code to select all rows for `patient_id` "P02" and only the columns `visit_date`, `updrs_iii`, and `ledd_mg`. 
2. Write code to select show me the last 3 rows for columns `patient_id`, `visit_date`, and `medication` using first .tail() and then .loc[]
3. Can you find which gender is the patient with zero `ledd_mg` using a query?
4. For all female patients, display all the columns that contain "medication" in their name.


In [ ]:
# Write your code here

## 🧩 1️⃣ Understanding Data Types in pandas

### Why it matters

In EHR data, each column stores a **different type of information**:

* **Numeric:** clinical scores (e.g. `updrs_iii`, `ledd_mg`)
* **Categorical:** sex, medication type
* **Datetime:** visit or birth dates
* **String/object:** free text (e.g., clinician notes)

Getting types right lets pandas apply the right operations (e.g., compute means for numbers, sort by date, count categories, etc.).


### Let's check your data types!

In [ ]:
df.dtypes

> This provides you with a quick overview of how pandas “understands” each column.

### 🧮 1️⃣ Numeric types — `int64` and `float64`

* **What they are:** Whole numbers (`int64`) or decimals (`float64`).
* **Used for:** Quantitative, measurable values that you can perform arithmetic on (add, subtract, divide, compute means, etc.).
* **Examples in your dataset:**

  * `ledd_mg`: medication dose in mg (integer or float)
  * `updrs_iii`: motor severity score (float because it contains `NaN` missing values)

> 💡 **Note:** In pandas, if any value in a numeric column is missing (`NaN`), pandas automatically upgrades the column to `float64`, since `NaN` is a floating-point concept.

**Typical operations:**


In [ ]:
print("Average dose (mg):", df["ledd_mg"].mean())   # average dose

Check if BMI is numeric! If not, convert it and verify by running `df.dtypes` again.



In [ ]:
df["BMI"] = pd.to_numeric(df["BMI"], errors="coerce")  # convert BMI to numeric, coerce errors to NaN

In [ ]:
df["BMI"] 

In [ ]:
df["updrs_iii"].max()    # highest severity

In [ ]:
df["ledd_mg"] / 1000     # convert mg to g

### Let's dive into the available operations a bit deeper!

#### 🔢 Basic arithmetic operations on numeric columns

Imagine we have a dataframe with clinical scores and medication doses just for one patient:

```python
df = pd.DataFrame({
    "patient_id": ["P01"] * 5,
    "age": [65, 65, 66, 66, 67],
    "BMI": [22.53, 22.28, 22.67, 24.02, 24.72],
    "visit_date": pd.to_datetime(["2024-01-15", "2024-06-20", 
                                "2024-12-05", "2025-03-10", 
                                "2025-09-15"]),
    "updrs_iii": [22.0, 25.5, 30.0, 28.0, 35.0],
    "ledd_mg": [300, 400, 500, 600, 700]
})
```

Pandas supports a variety of arithmetic operations that can help us with analysis!

| Operation      | Example                       | Meaning                               |
| -------------- | ----------------------------- | ------------------------------------- |
| Addition       | `df["updrs_iii"] + 5`         | Increase all UPDRS scores by 5 points |
| Subtraction    | `df["age"] - 60`              | Difference from 60 years old          |
| Multiplication | `df["ledd_mg"] * 0.001`       | Convert mg to g                       |
| Division       | `df["ledd_mg"] / df["BMI"]`   | LEDD normalised by BMI                |
| Power          | `df["age"] ** 2`              | Square of age                         |
| Rounding       | `df["BMI"].round(1)`          | Round to 1 decimal                    |
| Absolute value | `df["updrs_iii"].abs()`       | Remove negative sign (if any)         |
| Cumulative sum | `df["ledd_mg"].cumsum()`      | Running total of LEDD exposure                       |
| Difference     | `df["ledd_mg"].diff()`        | Change between two visits             |
| Ranking        | `df["updrs_iii"].rank()`      | Patient rank by severity              |
| Sorting        | `df.sort_values("updrs_iii")` | Order by score                        |

> 💡 These work directly on pandas columns because they’re implemented **vector-wise** — the operation applies to each cell automatically.

---


#### 📈 Descriptive statistics (summary measures)

pandas makes it easy to calculate common summary statistics that describe a numeric column’s distribution.

| Statistic                   | Code                                          | Meaning                                                               |
| --------------------------- | --------------------------------------------- | --------------------------------------------------------------------- |
| **Mean** (average)          | `df["updrs_iii"].mean()`                      | The *centre* of the data — total divided by number of observations.   |
| **Median**                  | `df["updrs_iii"].median()`                    | The *middle* value when sorted; less sensitive to outliers.           |
| **Mode**                    | `df["updrs_iii"].mode()`                      | The *most frequent* value in the column.                              |
| **Standard deviation (SD)** | `df["updrs_iii"].std()`                       | Average distance of each value from the mean — a measure of *spread*. |
| **Variance**                | `df["updrs_iii"].var()`                       | SD squared; another measure of spread.                                |
| **Minimum / Maximum**       | `df["updrs_iii"].min()`, `.max()`             | Smallest and largest values.                                          |
| **Count (non-missing)**     | `df["updrs_iii"].count()`                     | Number of non-NaN values.                                             |
| **Sum**                     | `df["updrs_iii"].sum()`                       | Total of all values.                                                  |
| **Quantiles**               | `df["updrs_iii"].quantile([0.25, 0.5, 0.75])` | 25th, 50th (median), and 75th percentiles.                            |
| **Skewness**                | `df["updrs_iii"].skew()`                      | Whether the distribution is lopsided (tail to right or left).         |
| **Kurtosis**                | `df["updrs_iii"].kurt()`                      | How “peaked” or “flat” the distribution is.                           |


![mean_mode_median](https://tutor2u-net.imgix.net/subjects/geography/downloads/livestreams/studynoteimages/mean-median-mode.png?auto=compress%2Cformat&fit=clip&q=80&w=800)



> 📊 You can view many of these at once with:


In [ ]:
df[["updrs_iii", "ledd_mg"]].describe()

![mean_mode_median_meme](images/IMG_2884.WEBP)


---
### 🧠 Quick explanations of key statistical terms
---
#### Mean (Average)

The arithmetic mean is the **sum of all values divided by the number of observations**.

$
\text{Mean} = \frac{x_1 + x_2 + ... + x_n}{n}
$

Example:
If LEDD doses are 300, 500, and 700 mg → mean = (300 + 500 + 700)/3 = 500 mg.

Used when data are **symmetrically distributed**.

---

#### Median

The **middle** value once the data are sorted.

In  both discrete and continuous data, it works like this: if there are 9 patients, the median is the 5th when sorted by UPDRS score. For even counts, it’s the average of the two middle values.


✅ **Resistant to outliers**, so it’s better than the mean for skewed data.

---

#### Mode

The **most frequent** value in the data.
Useful for categorical or discrete numeric data (e.g., common UPDRS III test score, most frequent clinic).

---

#### Standard Deviation (SD)

Measures **spread or variability** — how far data points tend to deviate from the mean.

$
\text{SD} = \sqrt{\frac{\sum (x_i - \bar{x})^2}{n-1}}
$

* Low SD → most values close to the mean.
* High SD → data widely spread out.

In EHRs, SD helps gauge **heterogeneity** (e.g., how variable LEDD doses are between patients).

![](https://mathbitsnotebook.com/Algebra2/Statistics/normalpercentilegrph2.jpg)
---

#### Variance

The average of squared deviations from the mean.

$
\text{Variance} = \text{SD}^2
$

> Used mainly for statistical modelling; less intuitive than SD because it’s in squared units.

---

#### Quantiles & Percentiles

Divide data into equal parts:

* 25th percentile (Q1): 25 % of data below it
* 50th percentile (Q2): median
* 75th percentile (Q3): 75 % of data below it
* IQR = Q3 − Q1 measures spread of the middle 50 %.

![](https://i.sstatic.net/dTDGD.png)

---

#### Range

Maximum minus minimum — simplest measure of spread.

$
\text{Range} = \text{max} - \text{min}
$

---

#### Skewness

Indicates whether the data are **asymmetric**:

* **Right-skewed** (positive): tail to the right (e.g., LEDD — most low, few very high).
* **Left-skewed** (negative): tail to the left.

In discrete data, you can estimate skewness as Pearson's first coefficient of skewness (mode skewness):
$\text{Skewness} = \frac{\text{Mean} - \text{Mode}}{\text{SD}}$

In continuous data, you can quantify skewness as Pearson's second coefficient of skewness (median skewness):
$\text{Skewness} = \frac{3(\text{Mean} - \text{Median})}{\text{SD}}$    

![](https://www.statisticshowto.com/wp-content/uploads/2014/02/pearson-mode-skewness.jpg)
---
#### Kurtosis

Describes how “peaked” or “flat” the distribution is compared to a normal curve as well as how 'fat' the tails of the distribution are. High kurtosis → heavy tails (more outliers).

![](https://miro.medium.com/v2/resize:fit:1400/format:webp/1*JbVwjNfpkplE9G8Dnosqgw.jpeg )








### 🍎 <span style="color:red; font-weight:bold"> Student challenge  </span>

<span style="color:red; font-weight:bold">  Now, your turn! </span>

Calculate the mean, median, mode, standard deviation, variance, min, max, count, sum, 25th, 50th, and 75th percentiles, skew and kurtosis  for the `updrs_iii` column in the dataframe `df`. 

In [ ]:
# Write your code here

#### <span style="color:red"> 🍎 Bonus question! </span>

![](images/IMG_2876.JPG)

* Which of those cats have **higher standard deviation** than the other? Why?

* Which one has the **higher kurtosis**? Why? 

### 🔢 Combining numeric operations

You can combine methods easily:

In [ ]:
# Example summaries
mean_updrs = df["updrs_iii"].mean()
std_updrs = df["updrs_iii"].std()
cv_updrs = std_updrs / mean_updrs   # Coefficient of variation (relative spread)

print(f"Mean: {mean_updrs:.1f}, SD: {std_updrs:.1f}, CV: {cv_updrs:.2f}")

In [ ]:
# You can implement this using a single line!


# OR using a single line
cv_updrs = df["updrs_iii"].std() / df["updrs_iii"].mean()

print(f"CV: {cv_updrs:.2f}")


### 🧩 Creating New Columns and Rows in pandas

Once you understand **numeric data types** (`int64`, `float64`), you can start **building new variables** from existing ones — just like you’d do in Excel formulas, but faster and more reproducibly.

#### 🧮 1️⃣ Creating new columns

You can create a new column simply by assigning to a name inside square brackets `[]`.
```python

df["new_column_name"] = <some expression>
```
This adds a new column to your DataFrame.
The expression can be:

* A **constant** (e.g., `0`, `np.nan`, `1.5`)
* An **operation** on existing columns (addition, subtraction, division, etc.)
* A **function** applied to other columns



#### ➕ Add or subtract values

Let’s start with your existing numeric columns:
`ledd_mg`, `updrs_iii` 



In [ ]:
# Increase all LEDD doses by 100 mg
df["ledd_plus_100"] = df["ledd_mg"] + 100

# Difference between UPDRS score and 30
df["updrs_diff"] = df["updrs_iii"] - 30

In [ ]:
df[['ledd_mg','ledd_plus_100', 'updrs_iii', 'updrs_diff']]

#### ✖️ Multiply or divide columns

In [ ]:
# Convert LEDD from mg to grams
df["ledd_g"] = df["ledd_mg"] / 1000

# LEDD normalised by BMI (dose per body mass)
df["ledd_per_bmi"] = df["ledd_mg"] / df["BMI"]

df[['ledd_mg','ledd_g', 'BMI', 'ledd_per_bmi']]

#### ⚙️ Combine multiple operations

In [ ]:
# Adjust LEDD based on both BMI and severity
df["dose_index"] = (df["ledd_mg"] / df["BMI"]) * df["updrs_iii"]



> 💡 **Note:** pandas applies arithmetic “vectorised” — the operation happens automatically on every row.


### 🧩 3️⃣ Conditional columns (using `np.where` or boolean logic)

You can create new variables based on **conditions** — for example, flagging patients on medication or severe disease.

In [ ]:
import numpy as np

# 1 = on medication, 0 = no medication
df["on_medication"] = np.where(df["ledd_mg"] > 0, 1, 0)

# Flag severe UPDRS scores (≥ 59)
df["is_severe"] = np.where(df["updrs_iii"] >= 59, 1, 0)


You can also combine multiple conditions using `&` (AND), `|` (OR), and `~` (NOT):

In [ ]:
df["pd_like"] = np.where(
    (df["ledd_mg"] > 0) & (df["updrs_iii"] > 0),
    "Likely_PD",
    "Unlikely_PD"
)

###  4️⃣ Deleting columns
You can delete columns using `df.drop(columns=[...])` or `del df['column_name']`.


In [ ]:
# let's look at the columns we have in the dataframe
df.columns

In [ ]:
# let's create a list of columns to delete (all of the newly created ones)
columns_to_delete = ["ledd_plus_100", "updrs_diff", "ledd_g", "ledd_per_bmi", "dose_index", "on_medication", "is_severe", "pd_like"]

# delete the columns using df.drop()
df = df.drop(columns=columns_to_delete)

In [ ]:
df.columns 

### ➕ 5️⃣ Adding new rows

You can append a single observation (row) using `pd.concat()` or `DataFrame.loc[]`.

In [ ]:
# Example: add one new patient visit (dictionary of values)
new_row = {
    "patient_id": "P05",
    "sex": "Male",
    "ledd_mg": 550,
    "updrs_iii": 42,
    "BMI": 26.4,
    "date_of_birth": "10.10.1960",
    "visit_date": "2024-05-01",
    "medication": "Entacapone 200",
}
df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)

Or modify a specific row directly:

In [ ]:
df.loc[len(df)] = ["P06", "Unidentified", "22.05.1980", 28, 400, "2025-01-05", "Levodopa", "No", 23.4, "09/2025"]

> ⚠️ **Tip:** Avoid manual row addition in real analyses — in EHR data, rows usually come from merging or concatenating datasets.







### 🧠 6️⃣ Common numeric column operations for new columns

| Operation       | Code                        | Description                        |
| --------------- | --------------------------- | ---------------------------------- |
| Addition        | `df["A"] + df["B"]`         | Sum two columns                    |
| Subtraction     | `df["A"] - df["B"]`         | Difference between columns         |
| Multiplication  | `df["A"] * 10`              | Scale values                       |
| Division        | `df["A"] / df["B"]`         | Create ratio                       |
| Modulus         | `df["A"] % 2`               | Remainder after division           |
| Power           | `df["A"] ** 2`              | Square or cube values              |
| Logarithm       | `np.log(df["A"])`           | Useful for skewed data (e.g. LEDD) |
| Rounding        | `df["A"].round(1)`          | Round numeric results              |
| Type conversion | `df["A"].astype("float64")` | Change numeric type                |

---



### 🍎 <span style="color:red; font-weight:bold">Student challenge 


 <span style="color:red; font-weight:bold">Now your turn!</span>

  <span style="color:red; font-weight:bold"> Use the dataset `df` to explore numeric data and create a few new variables.</span>

**Tasks:**

1. Calculate the **mean**, **median**, and **standard deviation** of `ledd_mg`. 

2. Compare the **mean** and **median** — does this suggest LEDD is *skewed*? Is it right-skewed or left-skewed? Get a value for **skewness** using both Pearson's formula and `.skew()`.

3. Compute the **interquartile range (IQR)** for `updrs_iii` (Q3 − Q1) as a variable `iqr_updrs`.

4. Create a new column `ledd_per_updrs` = LEDD ÷ UPDRS III.

5. Create a Boolean column `high_bmi` that is `True` if BMI > 30 (obesity threshold).

6. One way to deal with the skewness of LEDD is to standardise it using a ** modified z-score**. In comparison to conventional z-score, which measures how far the value is from the mean in terms of standard deviations, modified z-score uses median and median absolute deviation (MAD), making it usable in skewed distributions.
Create a new column `ledd_zscore` using the formula: 

    $ ledd\_zscore = 0.6745 \times \frac{ledd\_mg - median(ledd\_mg)}{MAD(ledd\_mg)}$

    where $MAD(ledd\_mg) = median(|ledd\_mg - median(ledd\_mg)|)$
    
    (Hint: use and `np.abs()` functions)






In [ ]:
# Write your code here 

## Other types of data structures in pandas


### 🧩 2️⃣ Boolean type — `bool`

We've already encountered Boolean columns when we created `high_bmi` above.
* **What it is:** Logical True/False values.
* **Used for:** Flags or binary indicators (e.g., has medication, abnormal result).
* **Examples:**
  You can *create* Boolean columns:



In [ ]:
df["is_on_medication"] = df["ledd_mg"] > 0

  This column will be `True` where dose > 0 and `False` otherwise.

In [ ]:
df[['ledd_mg', 'is_on_medication']]

### 🔤  Object / String type — `object` or `string`

* **What it is:** Free text or mixed data that pandas can’t classify automatically.
* **Used for:** Patient identifiers, notes, or descriptive text.
* **Examples:**

  * `patient_id`
  * `medication` (if not converted to category)




> ⚠️ `object` columns are flexible but slower for analysis — convert them to `category` or numeric if possible.

We can use methods such as `.unique()`, `.nunique()` and `.value_counts()` to explore unique values in an Object/String column:

In [ ]:
df["medication"].unique()

In [ ]:
df["medication"].nunique()            # count unique drug types

In [ ]:
df["medication"].value_counts()   # count occurrences of each drug type


> ⚠️ As you can see, we have 2 **occurences of Co-carbidopa** and 2 occurences of **Rotigotine**, but due to inconsistent capitalization and spacing, pandas treats them as **different values**. This is a common issue in EHR data entry, and it is important to **clean** such data before analysis using string methods.

#### 🔤 Common String Operations in pandas (object / string dtype)


Because a pandas column is a **Series** (a whole column, not a single string), we use the special `.str` accessor to apply text operations **element-wise** (to each cell).

> For example:


In [ ]:
df["medication"].str.lower()  

#### 🧮 Common `.str` methods and when to use them

| Method                                  | What it does                               | When to use                                                       |
| --------------------------------------- |  ------------------------------------------ | ----------------------------------------------------------------- |
| `.str.lower()`                          | Converts all text to lowercase             | Standardise case before comparison (`"Levodopa"` vs `"levodopa"`) |
| `.str.upper()`                          | Converts all letters to uppercase          | Formatting or matching uppercase codes                            |
| `.str.title()`                          | Capitalises first letter of each word      | Clean messy labels for presentation                               |
| `.str.strip()`                          | Removes leading/trailing spaces            | Clean extra whitespace from imported data                         |
| `.str.contains("levodopa", case=False)` |  Returns True if text contains substring    | Filter rows with certain keywords (e.g., `"Levodopa"`)            |
| `.str.replace("Co-", "")`               |  Replaces text patterns (supports regex)    | Clean prefixes/suffixes (e.g., “Co-”)                             |
| `.str.startswith("Levo")`               |  True/False if string starts with a pattern | Identify all Levodopa derivatives                                 |
| `.str.endswith("dopa")`                 |  True/False if ends with a pattern          | Detect variants of a drug ending in “dopa”                        |
| `.str.len()`                            |  Returns string length                      | Check length of IDs or text fields                                |
| `.str.slice(0,3)`                       |  Extract substring by position              | Get first 3 characters of `patient_id`                            |
| `.str.split(" ")`                       |  Splits a string into a list by spaces      | Separate multi-word fields (e.g., "Levodopa COMT")                |
| `.str.replace(r"\d+", "", regex=True)`  |  Remove all numbers using regex             | Clean IDs, strip version numbers                                  |
| `.str.cat(sep=", ")`                    |  Concatenate strings in a column            | Join text values for summary outputs                              |
| `.str.get(0)`                           |  Get character at a position                | Extract first letter of patient code                              |
| `.str.findall("pattern")`               | Return all regex matches per row           | Advanced pattern extraction (e.g., dose values from notes)        |

---

#### 🧠 Why use `.str`?

* Each cell is a string, but the column is a **pandas Series** — `.str` lets you broadcast string methods to *every row* at once.
* Without `.str`, normal Python string functions (like `.lower()`) won’t work on a whole column.


In [ ]:
# Find all patients on any dopa derivative formulation
mask = df["medication"].str.contains("dopa", case=False, na=False)
df[mask][["patient_id", "medication"]]


### 🍎 <span style="color:red; font-weight:bold">Student challenge 


 <span style="color:red; font-weight:bold">Now your turn!</span>

  <span style="color:red; font-weight:bold"> Pre-clean the dataset and explore medical prescription data.</span>

**Tasks:**

1. Using the `.str` methods, standardize the text in the `medication` column so that:
   - All entries are in lowercase.
   - Leading and trailing spaces are removed.
   - Any occurrence of "co-" is removed from the medication names.
   - Missing values are consistently represented as ""

2. Perform a new `.value_counts()` on the cleaned `medication` column to see how many unique medications there are and their frequencies.

3. Create new boolean column  'on_dopa_derivative' and 'on_dopamine_agonist' to mark patients on those medications 

4. Create new boolean column 'on_combination_therapy' to mark patients two or more medications prescribed (e.g. COMT with Dopa Derivative)


In [ ]:
# Write your code here


#### 💡 Tips

If you have messy text data:

1. Clean and standardise with `.str.lower()` and `.str.strip()`.
2. `.str.strip()` can also remove specific characters like punctuation or numbers, e.g.
    ```python
    df["column"].str.strip(".,;!0123")
    ```
2. Use `.str.contains()` or `.str.startswith()` for filtering.
3. Convert cleaned columns to `category` if values repeat often.
4. You can chain multiple `.str` methods together for more complex cleaning, e.g.:
    ```python
    df["column"].str.lower().str.strip().str.replace("old", "new")
    ```


### 🧬 4️⃣ Categorical types — `category`

* **What they are:** Columns that contain a limited number of possible values.
* **Used for:** Grouping and statistical comparisons — they are memory-efficient and improve performance.
* **Examples:**

  * `sex`: “Male”, “Female”
  * `other_medication`: “Yes”, “No”
  * `medication`: drug names (repeated categories)

You can convert to category with:

In [ ]:
df["sex"] = df["sex"].astype("category")

In [ ]:
df["sex"]

**Why use `category`:**

* Less memory use for repeated labels
* Helpful when modelling (encoded automatically as dummy variables)

### `groupby()` for categorical data and other types
You can use `.groupby()` to **aggregate** numeric data by categories.
For example, to compute the mean UPDRS III score by sex:


In [ ]:
# groupby() for UPDRS III by sex
df.groupby("sex")["updrs_iii"].mean()

The `.groupby()` method doesn't require the grouping column to be of type `category`, but converting it can speed up operations and reduce memory usage, especially with large datasets.

Let's try it on other_medication, which is currently of type object (string):


In [ ]:
df.groupby("other_medication")["updrs_iii"].mean()

You can also group by multiple columns, e.g., mean UPDRS III by sex and medication status:


In [ ]:
df.groupby(["sex", "other_medication"])["updrs_iii"].mean()


### 📅 3️⃣ Datetime type — `datetime64[ns]`

* **What it is:** Timestamps that include date (and optionally time) information.
* **Used for:** Age calculation, sorting visits, computing time differences, or visualising trends over time.
* **Examples:**

  * `date_of_birth`, `visit_date`, `id_last_update`

**Typical operations:**


In [ ]:
# turn the visit_date column to datetime
df["visit_date"] = pd.to_datetime(df["visit_date"])
df.dtypes

After converting date columns with `pd.to_datetime()`, you can perform many **time-based operations** to calculate differences, extract components, or resample over time.

In [ ]:
# Extract year or month
df["visit_year"] = df["visit_date"].dt.year
df["visit_month"] = df["visit_date"].dt.month

As you might remember, the 'id_last_update' is also a date, but it was logged in a different format. 
> `pd.to_datetime()` can parse many date formats automatically, but sometimes you need to **specify the format explicitly** using the `format` parameter. For example, if the date is in the format "MM/YYYY", you would use:


In [ ]:

df["id_last_update"] = pd.to_datetime(df["id_last_update"], format="%m/%Y")

In [ ]:

# Difference between two dates (today and last update)
df["days_since_update"] = (pd.Timestamp.now() - df["id_last_update"]).dt.days
df["days_since_update"]

In [ ]:
# Difference between two dates (today and last update) in months
df["months_since_update"] = (pd.Timestamp.now() - df["id_last_update"]).dt.days / 30
df["months_since_update"]

In [ ]:
# imagine we found out that the update dates were actually recorded 2 months, 28 days and 35 minutes later than they actually were

days_update = 2 * 30 + 28
df["id_last_update"] = df["id_last_update"] - pd.Timedelta(days=days_update, hours=0, minutes=35)

In [ ]:
df["id_last_update"]

In [ ]:
# Difference between two dates (today and last update) in years
df["years_since_update"] = (pd.Timestamp.now() - df["id_last_update"]).dt.days / 365.25
df["years_since_update"]



### ⏱️ Common Time & Date Operations in pandas

Here are some more examples of useful date/time operations you can perform in pandas:

| Operation                                    | Example code                                                                                     | Description                                                 | Typical use in clinical / longitudinal data          |
| -------------------------------------------- | ------------------------------------------------------------------------------------------------ | ----------------------------------------------------------- | ---------------------------------------------------- |
| **Days since last update**                   | `df["days_since_update"] = (pd.Timestamp.now() - df["id_last_update"]).dt.days`                  | Calculates number of days between the last update and today | Monitor data recency or stale patient records        |
| **Days between two events**                  | `df["days_between_visits"] = (df["visit_date"] - df["id_last_update"]).dt.days`                  | Difference between two date columns                         | Measure follow-up intervals or delays in visits      |
| **Age at visit (years)**                     | `df["age_at_visit"] = ((df["visit_date"] - df["date_of_birth"]).dt.days / 365.25).round(1)`      | Convert date difference to years                            | Calculate patient’s age dynamically at each visit    |
| **Extract year, month, day**                 | `df["visit_year"] = df["visit_date"].dt.year`<br>`df["visit_month"] = df["visit_date"].dt.month` | Extract components of the date                              | Group analyses by calendar year or seasonality       |
| **Weekday or weekend**                       | `df["visit_weekday"] = df["visit_date"].dt.day_name()`                                           | Returns full day name (Monday–Sunday)                       | Identify clinic patterns by day of the week          |
| **Time elapsed in months**                   | `df["months_since_dx"] = (df["years_since_dx"] * 12).round(0)`                                   | Converts years to months (if stored numerically)            | Harmonise units for monthly summaries                |
| **First and last recorded date per patient** | `df.groupby("patient_id")["visit_date"].agg(["min", "max"])`                                     | Finds each patient’s first and last visit date              | Identify earliest baseline and most recent follow-up |
| **Duration of follow-up (days)**             | `df["followup_duration"] = (df["max_visit"] - df["min_visit"]).dt.days`                          | Subtract first from last date                               | Measure longitudinal follow-up time                  |
| **Check for missing or future dates**        | `df.loc[df["visit_date"] > pd.Timestamp.now()]`                                                  | Find any visit recorded in the future                       | Data quality check for incorrect entry               |
| **Resampling by time period**                | `df.set_index("visit_date").resample("M")["ledd_mg"].mean()`                                     | Averages LEDD by month (resampling time series)             | Explore monthly trends in medication load            |
| **Rolling time windows**                     | `df.set_index("visit_date")["updrs_iii"].rolling("180D").mean()`                                 | Moving average over a 180-day window                        | Smooth longitudinal symptom trends                   |

---




### 💡 Tips 

* `.dt` is the **datetime accessor**, similar to `.str` for strings — it lets you access date parts and perform operations column-wise.
* `pd.Timestamp.now()` or `pd.Timestamp.today()` gives the current date/time.
* Differences between two `datetime64` columns produce a `timedelta64` series, from which you can extract `.dt.days`, etc.
* Always check for **time zone consistency** (`.dt.tz_localize()` / `.dt.tz_convert()`) in real-world datasets.

### ⚙️ 6️⃣ Missing data type — `NaN` (Not a Number)

* **What it is:** Placeholder for missing or undefined values.
* **Used for:** Representing blanks in numeric or datetime columns.
* **Handled by:** `pandas.isna()` or `.fillna()` for imputation later.

**Example:**

In [ ]:
df["updrs_iii"].isna().sum()   # count missing scores



```python

```

---

### 📊 Summary Table

| pandas dtype        | Description         | Typical use                            | Example column             |
| ------------------- | ------------------- | -------------------------------------- | -------------------------- |
| `int64`             | Integer numbers     | Counts, doses                          |  `updrs_iii`                 |
| `float64`           | Decimal numbers     | Scores with decimals or missing values |   `ledd_mg`              |
| `category`          | Repeated labels     | Sex, medication type                   | `sex`  |
| `datetime64[ns]`    | Date & time         | Visits, birthdates                     | `visit_date`               |
| `object` / `string` | Text                | IDs, free text                         | `patient_id`, `medication` |
| `bool`              | True/False          | Flags, conditions                      | Derived columns            |
| `NaN`               | Missing placeholder | Incomplete data                        | Gaps in `updrs_iii`        |



## ⚙️ Understanding `.transform()` and `lambda x:` in pandas

### 🧠 Why we need `.transform()`

When you use `groupby()`, pandas usually gives you **one value per group** — for example, the minimal LEDD per patient:



In [ ]:
df.groupby("patient_id")["ledd_mg"].min()


But sometimes we want to **transform** the group data **and return the result for every row** in the original DataFrame.

👉 That’s what **`.transform()`** does:
it performs a group-wise calculation but **keeps the same shape** as the original data.

#### 🔍 Example 1 — mean LEDD per patient, returned for every visit


In [ ]:
df["mean_ledd_per_patient"] = (
    df.groupby("patient_id")["ledd_mg"]
      .transform("min")
)
df[["patient_id", "ledd_mg", "mean_ledd_per_patient"]]


> Each visit now shows the patient’s own minimal LEDD.

> Compare this to `.groupby().min()`, which collapses the dataset to one row per patient.

This is useful when you want to **compare each observation to a group-level statistic** (e.g., how does current LEDD compare to the patient's smallest dose across observations?). We can easily compute this as a new column:

In [ ]:
df["ledd_now_vs_min"] = (
    df["ledd_mg"] - df.groupby("patient_id")["ledd_mg"].transform("min")
)
df[["patient_id", "ledd_mg", "mean_ledd_per_patient", "ledd_now_vs_min"]]

#### 🧮 Example 2 — using custom functions inside `.transform()`
You can also pass a **custom function** instead of a predefined string like `"mean"` or `"sum"`.

In [ ]:
def mad_zscore(series):
    # calculate MAD of LEDD
    mad_value = (series - series.median()).abs().median()
    return 0.6745 * (series - series.median()) / mad_value



df["ledd_mad_zscore_per_patient"] = (
    df.groupby("patient_id")["ledd_mg"].transform(mad_zscore)
)

df[["patient_id", "ledd_mg", "ledd_mad_zscore_per_patient"]]


> Here we standardise LEDD *within each patient* — the output keeps the same number of rows as the original dataset.


#### 🧩 What is `lambda x:` ?

A `lambda` is a **short, one-line function** used when you don’t want to define a full `def` block.

```python
lambda x: <expression using x>
```

* `x` represents the Series (the group of values passed in by pandas).
* Whatever the expression returns replaces the column values.

#### 💡 Example 3 — simple `lambda` transformations

In [ ]:
# Center LEDD values around the median
df["ledd_centered"] = df["ledd_mg"].transform(lambda x: x - x.median())

> You can also use logical conditions inside transform to filter out values to be transformed.

In [ ]:
# Add 10 to every LEDD value above 0
df["ledd_plus_10"] = df["ledd_mg"].transform(lambda x: x + 10 if x > 0 else x)

df[["patient_id", "ledd_mg", "ledd_plus_10"]]

In [ ]:
# Equivalently, you can have used
df["ledd_plus_10"] = df["ledd_mg"].transform(lambda x: 0 if x <= 0 else x + 10)

df[["patient_id", "ledd_mg", "ledd_plus_10"]]

When used with `groupby`, the `lambda` receives each group separately:

In [ ]:
df["ledd_centered_by_patient"] = (
    df.groupby("patient_id")["ledd_mg"]
      .transform(lambda x: x - x.median())
)
df[["patient_id", "ledd_mg", "ledd_centered_by_patient", "ledd_centered"]]

> In this case, the **centred LEDD for each patient** - `ledd_centred_by_patient` is calculated using the **patient-specific median**. The `ledd_centered`, on the other hand, uses the median group-wise median of **all patients**.


### ⚙️ Advanced Group Operations: `.transform()` vs `.agg()`

So far, you’ve learned that:

* `.groupby()` lets us split the data into groups (e.g., by `patient_id` or `sex`).
* `.transform()` applies an operation to each group **but keeps the same number of rows** as the original DataFrame.
* `lambda x:` lets us define custom mini-functions inline.

Now we’ll explore **`.agg()` (aggregate)** — another powerful pandas method for summarising data *by group*.


#### 🧠 What’s the difference?

| Method         | Output shape                    | Typical use                                                                           |
| -------------- | ------------------------------- | ------------------------------------------------------------------------------------- |
| `.transform()` | same number of rows as original | Create new columns (e.g., patient-level means or z-scores broadcast back to each row) |
| `.agg()`       | one row per group               | Create *summary tables* (e.g., mean, SD, min, max per patient)                        |



#### 🧩 Example 1 — Using `.agg()` for summary statistics per patient

In [ ]:
patient_summary = (
    df.groupby("patient_id")
      .agg({
          "ledd_mg": ["mean", "std", "min", "max"],
          "updrs_iii": ["mean", "min", "max", "count"]
      })
)
patient_summary.head()

This creates a **multi-index column** (hierarchical header) with your chosen statistics.

✅ Each row = **one patient**
✅ Columns = **summary measures** for that patient

> 💡 You can flatten column names if you prefer simple labels:

In [ ]:
patient_summary.columns = ['_'.join(col).strip() for col in patient_summary.columns.values]
patient_summary.reset_index(inplace=True)


In [ ]:
patient_summary.head()

#### 🧮 Example 2 — Applying multiple custom functions with `.agg()`

You can mix built-in functions, lambdas, and your own custom ones:

In [ ]:
def iqr(x):
    return x.quantile(0.75) - x.quantile(0.25)

patient_summary = (
    df.groupby("patient_id")
      .agg({
          "ledd_mg": ["mean", "std", iqr],
          "updrs_iii": [lambda x: x.mean(skipna=True), "count"]
      })
)

patient_summary.head()

> The `lambda x:` syntax lets you define one-line operations inline without writing a full `def` block.



#### ⚕️ Example 3 — Compare `.transform()` vs `.agg()`


In [ ]:
# .transform(): returns one value per row
df["mean_ledd_per_patient"] = df.groupby("patient_id")["ledd_mg"].transform("mean")
df[['patient_id', 'ledd_mg', 'mean_ledd_per_patient']]

In [ ]:

# .agg(): returns one row per patient
mean_ledd_per_patient = df.groupby("patient_id")["ledd_mg"].agg("mean").reset_index()
mean_ledd_per_patient

| Operation            | Returns                                 | Purpose                                      |
| -------------------- | --------------------------------------- | -------------------------------------------- |
| `.transform("mean")` | Same size as original DataFrame         | Adds a new column in the same table          |
| `.agg("mean")`       | Condensed DataFrame (one row per group) | Creates a summary table for further analysis |

---

#### 🔗 Example 4 — Merge the summary back into the main dataset

If you later want to use these statistics alongside the original data (for example, to compare each visit’s LEDD to the patient’s mean), you can merge them back on `patient_id`. 

P.S. we will cover more on merging and joining datasets in the later section of this tutorial!

In [ ]:
patient_summary.columns = ['_'.join(col).strip() for col in patient_summary.columns.values]
patient_summary.reset_index(inplace=True)

In [ ]:

df_new = df.merge(patient_summary, on="patient_id", how="left")
df_new



### 💡 Teaching tip

Use `.transform()` when you want to **add new columns** to the same DataFrame (same number of rows).
Use `.agg()` when you want to **create a separate summary DataFrame** (fewer rows).
Later in your course, these same operations become the basis for **statistical summaries**, **longitudinal modelling inputs**, and **data visualisation by group**.

### 🧩 Using `.apply()` in pandas

The `.apply()` method is one of pandas’ most flexible tools.
It lets you **run a custom function** on each element, column, or row of a DataFrame or Series.

### 🧠 What `.apply()` does

| Situation                         | Method                   | What it does                                            | Typical use                                     |
| --------------------------------- | ------------------------ | ------------------------------------------------------- | ----------------------------------------------- |
| **Series.apply(func)**            | `df["col"].apply(func)`  | Applies a function to **each value in a single column** | Clean or classify text (e.g., medication names) |
| **DataFrame.apply(func, axis=0)** | `df.apply(func)`         | Applies a function to each **column**                   | Compute column-wise stats                       |
| **DataFrame.apply(func, axis=1)** | `df.apply(func, axis=1)` | Applies a function to each **row**                      | Combine values from multiple columns            |


> 💡 Think of `.apply()` as a “bridge” between pandas and Python — it lets you use your own logic on pandas data.

#### 🔍 Example 1 — simple numeric function


In [ ]:
df["updrs_squared"] = df["updrs_iii"].apply(lambda x: x**2)
df[["updrs_iii", "updrs_squared"]].head()


Each value of `updrs_iii` is passed into the function `lambda x: x**2` and replaced with the result.

### 🔍 Example 2 — applying a custom function to text data

In [ ]:
def simplify_med_name(med):
    if pd.isna(med) or med == "":
        return "None"
    return med.lower().strip()

df["medication_simple_cleaning"] = df["medication"].apply(simplify_med_name)
df[["medication", "medication_simple_cleaning"]]


This runs your function on every value in the `medication` column — a **row-wise clean-up** step.

#### 🔢 Example 3 — using `.apply()` across rows

In [ ]:
def ledd_per_bmi(row):
    if pd.isna(row["ledd_mg"]) or pd.isna(row["BMI"]) or row["BMI"] == 0:
        return np.nan
    return row["ledd_mg"] / row["BMI"]

df["ledd_per_bmi"] = df.apply(ledd_per_bmi, axis=1)
df[["ledd_mg", "BMI", "ledd_per_bmi"]]

> 🧩 `axis=1` means “apply this function to each **row**”;
> each row is passed to your function as a small Series (`row["colname"]`).

### 🧩 Using `.map()` in pandas

The `.map()` method applies a **simple transformation** to each element of a pandas **Series** (i.e., one column).
It’s perfect for **value-by-value replacements** or simple logic, but less flexible than `.apply()`.

#### 🧠 When to use `.map()`

| Method        | Works on               | Purpose                                            |
| ------------- | ---------------------- | -------------------------------------------------- |
| `.map()`      | Series (single column) | Quick element-wise transformation or recoding      |
| `.apply()`    | Series or DataFrame    | More complex logic using custom functions          |
| `.applymap()` | Entire DataFrame       | Apply element-wise transformation across all cells |

> 💡 `.map()` is ideal when you need to **relabel**, **replace**, or **re-encode** column values using a mapping dictionary or a simple function.


#### 🔍 Example 1 — Re-label categorical values with a dictionary

In [ ]:
# Current column
df["sex"].head()

In [ ]:
# Map old values to new ones
sex_map = {"Male": "M", "Female": "F"}
df["sex_short"] = df["sex"].map(sex_map)



In [ ]:
df[["sex", "sex_short"]]


✅ **Result:**
Each value in `df["sex"]` is replaced with its mapped label.
Unmapped values become `NaN` (so always include all expected keys).


#### 🔍 Example 2 — Re-encode Yes/No answers numerically

In [ ]:
yn_map = {"Yes": 1, "No": 0}
df["other_medication_code"] = df["other_medication"].map(yn_map)
df[["other_medication", "other_medication_code"]].head()

✅ Converts text into numeric codes — useful before statistical tests or modelling.

#### 🔍 Example 3 — Use `.map()` with a simple function

In [ ]:
# Calculate dose group by numeric threshold
df["dose_group"] = df["ledd_mg"].map(lambda x: "High" if x > 600 else "Low")
df[["ledd_mg", "dose_group"]].head()

✅ `.map()` passes each value (`x`) into the lambda function and replaces it with the result.

> 🔹 Use this when the transformation depends on one variable.
> 🔹 For multi-column logic, prefer `.apply(axis=1)`.

#### 📊 Example 4 — Numeric mapping to standardised scores


In [ ]:
# Example: scale LEDD to g/day
df["ledd_g"] = df["ledd_mg"].map(lambda x: x / 1000 if pd.notna(x) else np.nan)

df[["ledd_mg", "ledd_g"]].head()

| Task                                      | Recommended method     | Example                                                   |
| ----------------------------------------- | ---------------------- | --------------------------------------------------------- |
| Replace or re-label fixed values          | `.map(dict)`           | `df["sex"].map({"Male": "M"})`                            |
| Apply simple 1-to-1 logic                 | `.map(lambda x: ...)`  | `df["ledd_mg"].map(lambda x: "High" if x>600 else "Low")` |
| Apply complex multi-column logic          | `.apply(func, axis=1)` | `df.apply(my_func, axis=1)`                               |
| Apply elementwise across entire DataFrame | `.applymap(func)`      | `df.applymap(lambda x: str(x).lower())`                   |

---


## 🧩 Binning Continuous Data with `pd.cut()`

### 🧠 What is `pd.cut()` and `pd.qcut()`?

`pd.cut()` converts **continuous numeric variables** (like age, BMI, or UPDRS scores) into **discrete categories or “bins”**.

`pd.qcut()` does the same but creates bins with **equal numbers of observations** (quantiles).

This process is called **binning** or **discretisation**.

It’s useful when you want to:

* group continuous data into ranges (e.g., *young*, *middle-aged*, *elderly*);
* count how many patients fall in each range;
* compare distributions visually (bar plots, cross-tabulations);
* prepare data for categorical analysis (e.g., t-tests by severity group).


### 🔢 Example — Bin `age` into age groups

In [ ]:
BMI_groups = [0, 18.5, 24.9, 25, 30, 35, 40]
BMI_labels = ["underweight", "normal weight", "overweight", "obesity I", "obesity II", "obesity III"]

df["bmi_group"] = pd.cut(df["BMI"], bins=BMI_groups, labels=BMI_labels, right=True)
df[["patient_id", "BMI", "bmi_group"]].head(8)

**Explanation:**

* `bins` define numeric intervals (here: 0–39, 40–59, etc.).
* `labels` assign readable names to each range.
* `right=False` means the intervals are *left-closed, right-open* — e.g., 40 ≤ age < 60.

> 💡 You can omit `labels` if you just want the numeric intervals as output.

### 🎚️ Example — Automatic equal-width or quantile bins

#### Equal-width bins:

In [ ]:
df["ledd_band"] = pd.cut(df["ledd_mg"], bins=4)
df["ledd_band"].value_counts()

Divides LEDD into **4 equal-width intervals** based on its numeric range.

#### Quantile-based bins (equal-sized groups):

In [ ]:
df["ledd_quartile"] = pd.qcut(df["ledd_mg"], q=4)
df["ledd_quartile"].value_counts()

Divides LEDD into **quartiles** (each bin has ~25% of data).
Useful for comparing “lowest-dose” vs “highest-dose” patients.

#### 🧮 Example — Combine `pd.cut()` with `.groupby()`
You can group by the new bins to calculate statistics for each range:

In [ ]:
df.groupby("bmi_group")["updrs_iii"].mean().round(1)



#### ⚙️ Other useful parameters

| Parameter         | Description                          | Example                         |
| ----------------- | ------------------------------------ | ------------------------------- |
| `bins=`           | Number of bins or list of cut points | `bins=5` or `bins=[0,10,20,30]` |
| `labels=`         | Custom labels for bins               | `labels=["low","mid","high"]`   |
| `right=`          | Include the right edge in intervals  | `right=False` (default is True) |
| `include_lowest=` | Include lowest value in first bin    | `include_lowest=True`           |
| `precision=`      | Decimal precision for bin edges      | `precision=1`                   |
| `duplicates=`     | Handles duplicate bin edges          | `duplicates='drop'`             |

---




## 📚 Unified Summary: pandas Data Transformation, Aggregation & Binning Methods

| Method / Pattern                               | Works on                   | Output shape                | Typical purpose                                                 | When to use                                                                  | Example                                                                                       |
| ---------------------------------------------- | -------------------------- | --------------------------- | --------------------------------------------------------------- | ---------------------------------------------------------------------------- | --------------------------------------------------------------------------------------------- |
| **`.map()`**                                   | Series (one column)        | ✅ Same length               | Simple element-wise transformation using a dictionary or lambda | Quick relabelling or 1-to-1 mapping of values                                | `df["sex"].map({"Male": "M", "Female": "F"})`                                                 |
| **`.map(lambda x: …)`**                        | Series                     | ✅ Same length               | Apply short logic to each element                               | Simple condition using a single column                                       | `df["ledd_mg"].map(lambda x: "High" if x>600 else "Low")`                                     |
| **`.apply(func)`**                             | Series / DataFrame         | ✅ Same length               | Apply a custom function element-wise or row-wise                | When logic depends on multiple columns or more complex rules                 | `df.apply(my_func, axis=1)`                                                                   |
| **`.applymap(func)`**                          | Entire DataFrame           | ✅ Same length               | Apply an element-wise function to every cell                    | Clean or format all values at once                                           | `df.applymap(lambda x: str(x).lower())`                                                       |
| **`.groupby().transform(func)`**               | Grouped Series             | ✅ Same length               | Apply a function *by group* and broadcast result back           | Derive new columns (e.g., per-patient mean, z-scores)                        | `df.groupby("patient_id")["ledd_mg"].transform("mean")`                                       |
| **`.groupby().transform(lambda x: …)`**        | Grouped Series             | ✅ Same length               | Apply custom group-wise logic                                   | Complex per-group operations (e.g., rolling mean, scaling)                   | `df.groupby("patient_id")["ledd_mg"].transform(lambda x: x/x.max())`                          |
| **`.groupby().agg(func)`**                     | Grouped Series / DataFrame | ❌ One row per group         | Compute summary statistics per group                            | Summarise data into a smaller table (per-patient or per-clinic)              | `df.groupby("patient_id")["ledd_mg"].agg(["mean","max","std"])`                               |
| **`.agg(["mean", "max", lambda x: x.std()])`** | Grouped DataFrame          | ❌ One row per group         | Combine built-in and custom stats in one command                | Create descriptive summary reports                                           | `df.groupby("patient_id").agg({"ledd_mg":["mean","std"], "updrs_iii":["mean","count"]})`      |
| **`.transform("mean")`**                       | Grouped Series             | ✅ Same length               | Add group summaries back into original data                     | To attach group averages to every row                                        | `df["mean_ledd_per_patient"] = df.groupby("patient_id")["ledd_mg"].transform("mean")`         |
| **`.apply(lambda x: x.rolling(...).mean())`**  | Series / Grouped Series    | ✅ Same length               | Apply rolling or custom window operations                       | Longitudinal smoothing, time-based metrics                                   | `df.groupby("patient_id")["ledd_mg"].transform(lambda x: x.rolling(365,1).mean())`            |
| **`.agg()` + `reset_index()`**                 | Grouped DataFrame          | ❌ Reduced table             | Store group-level summaries in a separate DataFrame             | When you need a summary table for further analysis or merging                | `patient_summary = df.groupby("patient_id").agg(mean_ledd=("ledd_mg","mean")).reset_index()`  |
| **`pd.cut()`**                                 | Numeric Series             | ✅ Same length (categorical) | Bin continuous variables into fixed numeric ranges              | Create categorical groups from continuous data (e.g., BMI, age, UPDRS bands) | `pd.cut(df["BMI"], [0,18.5,25,30,100], labels=["Underweight","Normal","Overweight","Obese"])` |
| **`pd.qcut()`**                                | Numeric Series             | ✅ Same length (categorical) | Bin continuous variables into quantiles (equal-sized groups)    | Divide continuous data into quartiles, deciles, etc.                         | `pd.qcut(df["ledd_mg"], q=4, labels=["Q1","Q2","Q3","Q4"])`                                   |

---

### 🧭 Quick Guide to Choosing the Right Method

| Goal                                                   | Recommended method                            |
| ------------------------------------------------------ | --------------------------------------------- |
| Replace or recode single-column values                 | `.map(dict)` or `.map(lambda x: …)`           |
| Apply conditional logic involving several columns      | `.apply(func, axis=1)`                        |
| Clean or format *every* cell in a DataFrame            | `.applymap(func)`                             |
| Add derived columns calculated per group (same size)   | `.groupby().transform()`                      |
| Compute summary statistics per group (fewer rows)      | `.groupby().agg()`                            |
| Mix built-in and custom aggregations                   | `.groupby().agg(["mean", lambda x: x.std()])` |
| Compute rolling or windowed metrics                    | `.transform(lambda x: x.rolling(...).mean())` |
| **Convert continuous variables into categorical bins** | **`pd.cut()` or `pd.qcut()`**                 |

---




### 📚 Unified Summary: pandas Data Transformation & Aggregation Methods

| Method / Pattern                               | Works on                   | Output shape        | Typical purpose                                                 | When to use                                                     | Example                                                                                      |
| ---------------------------------------------- | -------------------------- | ------------------- | --------------------------------------------------------------- | --------------------------------------------------------------- | -------------------------------------------------------------------------------------------- |
| **`.map()`**                                   | Series (one column)        | ✅ Same length       | Simple element-wise transformation using a dictionary or lambda | Quick relabelling or 1-to-1 mapping of values                   | `df["sex"].map({"Male": "M", "Female": "F"})`                                                |
| **`.map(lambda x: …)`**                        | Series                     | ✅ Same length       | Apply short logic to each element                               | Simple condition using a single column                          | `df["ledd_mg"].map(lambda x: "High" if x>600 else "Low")`                                    |
| **`.apply(func)`**                             | Series / DataFrame         | ✅ Same length       | Apply a custom function element-wise or row-wise                | When logic depends on multiple columns or more complex rules    | `df.apply(my_func, axis=1)`                                                                  |
| **`.applymap(func)`**                          | Entire DataFrame           | ✅ Same length       | Apply an element-wise function to every cell                    | Clean or format all values at once                              | `df.applymap(lambda x: str(x).lower())`                                                      |
| **`.groupby().transform(func)`**               | Grouped Series             | ✅ Same length       | Apply a function *by group* and broadcast result back           | Derive new columns (e.g., per-patient mean, z-scores)           | `df.groupby("patient_id")["ledd_mg"].transform("mean")`                                      |
| **`.groupby().transform(lambda x: …)`**        | Grouped Series             | ✅ Same length       | Apply custom group-wise logic                                   | Complex per-group operations (e.g., rolling mean, scaling)      | `df.groupby("patient_id")["ledd_mg"].transform(lambda x: x/x.max())`                         |
| **`.groupby().agg(func)`**                     | Grouped Series / DataFrame | ❌ One row per group | Compute summary statistics per group                            | Summarise data into a smaller table (per-patient or per-clinic) | `df.groupby("patient_id")["ledd_mg"].agg(["mean","max","std"])`                              |
| **`.agg(["mean", "max", lambda x: x.std()])`** | Grouped DataFrame          | ❌ One row per group | Combine built-in and custom stats in one command                | Create descriptive summary reports                              | `df.groupby("patient_id").agg({"ledd_mg":["mean","std"], "updrs_iii":["mean","count"]})`     |
| **`.transform("mean")`**                       | Grouped Series             | ✅ Same length       | Add group summaries back into original data                     | To attach group averages to every row                           | `df["mean_ledd_per_patient"] = df.groupby("patient_id")["ledd_mg"].transform("mean")`        |
| **`.apply(lambda x: x.rolling(...).mean())`**  | Series / Grouped Series    | ✅ Same length       | Apply rolling or custom window operations                       | Longitudinal smoothing, time-based metrics                      | `df.groupby("patient_id")["ledd_mg"].transform(lambda x: x.rolling(365,1).mean())`           |
| **`.agg()` + `reset_index()`**                 | Grouped DataFrame          | ❌ Reduced table     | Store group-level summaries in a separate DataFrame             | When you need a summary table for further analysis or merging   | `patient_summary = df.groupby("patient_id").agg(mean_ledd=("ledd_mg","mean")).reset_index()` |



### 🧭 Quick guide to choosing the right method

| Goal                                                 | Recommended method                            |
| ---------------------------------------------------- | --------------------------------------------- |
| Replace or recode single-column values               | `.map(dict)` or `.map(lambda x: …)`           |
| Apply conditional logic involving several columns    | `.apply(func, axis=1)`                        |
| Clean or format *every* cell in a DataFrame          | `.applymap(func)`                             |
| Add derived columns calculated per group (same size) | `.groupby().transform()`                      |
| Compute summary statistics per group (fewer rows)    | `.groupby().agg()`                            |
| Mix built-in and custom aggregations                 | `.groupby().agg(["mean", lambda x: x.std()])` |
| Compute rolling or windowed metrics                  | `.transform(lambda x: x.rolling(...).mean())` |







### 💡 Teaching tip

When deciding between `.map()`, `.apply()`, `.transform()`, and `.agg()`, ask yourself:

1. **Do I want the same number of rows back?**
   → Yes → use `.map()`, `.apply()`, or `.transform()`.
   → No (summary table) → use `.agg()`.

2. **Is my logic simple or multi-column?**
   → One column → `.map()` or `.apply()` on a Series.
   → Several columns → `.apply(axis=1)`.

---



### 🧪 <span style="color:red; font-weight:bold">Your Turn!</span>

1. Use `.map()` to recode `sex` into `"Male" → 1`, `"Female" → 0`.
2. Map `other_medication` ("Yes"/"No") into Boolean `True`/`False`.

4. Check your results using `.value_counts()` for each new column.

---


### 🍎 <span style="color:red; font-weight:bold">Student challenge 

<span style="color:red; font-weight:bold">Now your turn!</span>
>
> 1. Using the dataset `df`, convert the `date_of_birth` and `visit_date` columns to datetime format if they are not already.
> 2. Calculate the patien|t’s **age at last visit**: `visit_date − date_of_birth`.
> 3. Create a new column `updrs_iii_diag` where updrs <32  is 'No PD' and >=32 is 'PD' using .map().
> 4. Create a new category column from pre-cleaned 'medication' column, called `medication_type`, with the following categories:
>    - 'Dopamine Agonist' if medication contains 'rotigotine', 'pramipexole', or 'ropinirole'
>    - 'Levodopa Derivative' if medication contains 'levodopa' or 'carbidopa' or 'co-beneldopa'
>    - 'COMT Inhibitor' if medication contains 'Entacapone' or 'Tolcapone'
>    - 'COMT mixture' if medication contains both 'Levodopa Derivative' and 'COMT Inhibitor'
>    - 'Other' for all other medications
>    - 'None' if no medication is prescribed (empty string)
> 5. Create a column medication_abbreviation with the following mappings:
>    - 'Dopamine Agonist' → 'DA'
>    - 'Levodopa Derivative' → 'LD'
>    - 'COMT Inhibitor' → 'COMT'
>    - 'COMT mixture' → 'COMT+LD'
>    - 'Other' → 'Other'
>    - 'None' → 'None'
> 6. For each category in 'medication_type', use `.groupby()` to calculate the mean and standard deviation of `ledd_mg` and `updrs_iii`.
> 7. Create a new category column `age_group` with bins: '<60', '60-69', '70-79', '80+' based on `age_at_visit`.
> 8. Use `.groupby()` to calculate the mean `updrs_iii` score for each `age_group` and `sex` combination.
> 9. Calculate the rolling window average of ledd over a 365-day period for each patient. Create a new column `ledd_rolling_avg_365` to store this value.
> 10. Compute the mean and SD of `updrs_iii` per patient using `.agg()` and save as `patient_summary`. Merge `patient_summary` back into `df` on `patient_id`.
> 11. Create `updrs_zscore` = (`updrs_iii` − mean) / SD using `.transform()` or `.apply(lambda x: …)`. Count how many visits have `updrs_zscore > 0.5` (above-average motor severity).


In [ ]:
# Write your code here 


## 🔗 Merging / Joining Datasets in pandas

In real-world healthcare analytics, data often lives in **multiple tables**:

* 🧍‍♀️ **Patients table** — demographics, IDs, scores
* 🏥 **Clinics table** — clinic location, region, deprivation index
* 💊 **Medication or visits table** — prescriptions, doses, visit dates

To analyse the data together, you must **merge** (or “join”) these tables into one.

pandas’ `merge()` works similarly to **SQL joins**, letting you combine datasets based on shared columns (keys).

### ⚙️ Syntax
```python
pd.merge(left, right, on="key_column", how="join_type")
```

| Parameter       | Description                                                |
| --------------- | ---------------------------------------------------------- |
| `left`, `right` | DataFrames you want to merge                               |
| `on`            | Column(s) they share in common (the “key”)                 |
| `how`           | Type of join: `'inner'`, `'left'`, `'right'`, or `'outer'` |



---
### 🧠 Types of merges and when to use them

| Join type   | Keeps rows from…                                                   | Description                                                                    | When to use (clinical example)                                                                             |
| ----------- | ------------------------------------------------------------------ | ------------------------------------------------------------------------------ | ---------------------------------------------------------------------------------------------------------- |
| **`inner`** | Only rows with matching keys in *both* tables                      | Keeps **intersection** — only records present in both DataFrames               | When you only want patients who appear in both datasets (e.g., only those with both visit and clinic info) |
| **`left`**  | All rows from the **left** DataFrame, matching info from the right | Keeps **everything from your main dataset**; fills unmatched values with `NaN` | Most common: when patient-level data is primary and you want to add clinic info (even if some don’t match) |
| **`right`** | All rows from the **right** DataFrame, matching info from the left | Mirror of left join — keeps all right rows                                     | When clinic data is primary and you only care about clinics even if some have no patients                  |
| **`outer`** | All rows from **both** DataFrames                                  | Keeps **union** — everything, matched or not                                   | When you want a full overview of all patients and all clinics, even if some don’t overlap                  |
| **`cross`** | All combinations (cartesian product)                               | Every row in left joins with every row in right                                | Rarely used in EHR data; useful for simulation or all-pairs comparisons                                    |

---


### 🔍 Visual summary (conceptual)

Imagine merging two datasets by `patient_id`:

```
df (patients):          extra_info (extra)
P01  ✓                  P01  ✓
P02  ✓                  P02  ✓
P03  ✓                  P05  ✓
P04  ✓                  
```

| Join type | Result                                               |
| --------- | ---------------------------------------------------- |
| `inner`   | P01, P02, P03  → Only matches present in both        |
| `left`    | P01, P02, P03, P04 → Keeps all from df, adds matches |
| `right`   | P01, P02, P03, P05 → Keeps all from extra_info       |
| `outer`   | P01, P02, P03, P04, P05 → Everything                 |

![](images/IMG_2880.JPG)

### 🧩 Example: Two supplementary tables

In [ ]:
# Table 1: additional patient info
extra_info = pd.DataFrame({
    "patient_id": ["P01", "P02", "P03", "P04", "P07"],
    "height_cm": [172, 160, 158, 182 , 175],
    "weight_kg": [72, 65, 60, 80, 70],
    "smoking_status": ["Former", "Never", "Current", "Never", "Former"],
    "clinic_id": [101, 102, 101, 103, 102],
})

# Table 2: clinic information
clinics = pd.DataFrame({
    "clinic_id": [101, 102, 103],
    "clinic_name": ["St. Mary’s", "Queen Elizabeth", "Royal Infirmary"],
    "deprivation_index": [0.25, 0.52, 0.41],
    "region": ["London", "Birmingham", "Manchester"]
})


### 🔗 Merge examples
---

#### 1️⃣ Left join (most common)

Keep all patient rows; add extra info where available.

In [ ]:
df_merged = df.merge(extra_info, on="patient_id", how="left")
df_merged 

In [ ]:
df_merged["patient_id"].unique()

✅ Keeps all patients, even if some have no matching `extra_info`.
Unmatched fields become `NaN`.

#### 2️⃣ Inner join

Keep only patients who exist in *both* `df` and `extra_info`.

In [ ]:
df_inner = df.merge(extra_info, on="patient_id", how="inner")
df_inner

In [ ]:
df_inner["patient_id"].unique()



✅ Removes any patients not found in both — useful for analysis requiring complete records.

---
#### 3️⃣ Right join

Keep all rows from `extra_info`, even if some patients aren’t in `df`.

In [ ]:
df_right = df.merge(extra_info, on="patient_id", how="right")
df_right

In [ ]:
df_right["patient_id"].unique()


✅ Useful when you want to check which patients in a registry are missing from your primary dataset.

---

#### 4️⃣ Outer join

Keep *everything* from both sides.

In [ ]:
df_outer = df.merge(extra_info, on="patient_id", how="outer")
df_outer

In [ ]:
df_outer["patient_id"].unique()


✅ Keeps all patients — fills missing columns with `NaN`.
Great for **data audits** to find mismatched IDs.

---

#### 5️⃣ Merge on a different key

Once you’ve added `clinic_id` to patients, you can merge with the clinic table:

In [ ]:
df_merged = df_merged.merge(clinics, on="clinic_id", how="left")
df_merged


Now you can analyse LEDD or UPDRS by **clinic region** or **deprivation index**.

### 🧠 When to use which

| Goal                                              | Recommended join |
| ------------------------------------------------- | ---------------- |
| Add extra information to your *main* dataset      | `how="left"`     |
| Keep only records that appear in both             | `how="inner"`    |
| Investigate missing patients or unmatched records | `how="outer"`    |
| Keep all rows from supplementary table            | `how="right"`    |
| Generate all combinations (rare)                  | `how="cross"`    |

---


### 💡 Teaching tip

* Always ask yourself: *“Which dataset do I want to keep completely?”* → that’s your **left** DataFrame.
* Use **left joins** most of the time in EHR work — they preserve the full clinical cohort and simply add metadata from other tables.
* Reserve **outer joins** for *data validation or reconciliation* tasks (e.g., checking which patient IDs failed to match).



### 🍎 <span style="color:red; font-weight:bold">Student challenge 

 <span style="color:red; font-weight:bold">Now your turn!</span>

> 1. Merge the `extra_info` DataFrame into `df` using a **left join**.
> 2. Merge the `clinics` DataFrame into your merged dataset using `clinic_id`.
> 3. Create a summary table showing the **average BMI** and **mean LEDD** by `region`.
> 4. Identify any patients who don’t have a matching `clinic_id` (hint: `df_merged["clinic_id"].isna()`).
> 5. (Challenge) Perform an **outer join** between `df` and `extra_info` and count how many unmatched IDs exist in each dataset.


In [ ]:
# Write your code here